In [3]:
# Demonstrate the main Royal Care Clinic workflow.

from datetime import datetime, timedelta  # Import date/time utilities.

from clinic import Clinic, Department  # Import clinic classes.
from users import Patient, Doctor  # Import patient and doctor classes.
from scheduling import Appointment  # Import appointment class.
from medical import (  # Import medical classes.
    MedicalVisit,
    Prescription,
    PrescribedMedicine,
    LabOrder,
    LabResult,
)
from billing import Invoice, Payment, InsuranceClaim  # Import billing classes.


# Create the clinic.
clinic = Clinic(
    1, "Royal Care Clinic", "UAE",
    "+971 12 345 5678", "info@royalcare.example", "https://royalcare.example"
)

# Create a department and add it to the clinic.
department = Department(1, "General Practice", "09:00-17:00", "03001112222")
clinic.add_department(department)

# Register a patient.
patient = Patient(
    101, "Ali", "1998-04-12", "+971 12 345 5678",
    "ali@example.com", "Lahore", "+971 12 345 5678",
    "Example Health Insurance", ["Penicillin"], "No additional history recorded."
)

# Register a doctor.
doctor = Doctor(
    201, "Dr. Ahmad", "+971 12 345 1234", "Ahmad@example.com",
    "09:00-17:00", "General Practitioner", "MBBS", 1500.00
)

# Schedule an appointment for the next Monday.
today = datetime.now()  # Retrieve the current date and time.
days_until_monday = (7 - today.weekday()) % 7 or 7  # Calculate next Monday.
next_monday = (today + timedelta(days=days_until_monday)).replace(
    hour=10, minute=0, second=0, microsecond=0
)  # Set appointment time to 10 AM.

appointment = Appointment(
    301, patient, doctor, next_monday, 30, "General consultation"
)  # Create the appointment.

patient.add_appointment(appointment)  # Link appointment to patient.
appointment.confirm()  # Confirm the booking.

# Record the completed consultation.
visit = MedicalVisit(
    401, datetime.now(), "Seasonal allergy",
    "Patient reported sneezing and mild nasal symptoms."
)  # Create a medical visit.

visit.add_vital("Temperature", "36.8 C")  # Record temperature.
visit.add_vital("Blood Pressure", "118/76 mmHg")  # Record blood pressure.
visit.get_record().add_attachment("visit-summary-reference.pdf")  # Add document reference.
appointment.set_visit(visit)  # Link the visit to the appointment.
appointment.set_status("COMPLETED")  # Mark the appointment completed.

# Create a prescription and add a sample medicine item.
prescription = Prescription(501)  # Create prescription.
medicine = PrescribedMedicine(
    "Example medicine", "As prescribed", "Follow doctor's instructions", 1
)  # Use a placeholder item for demonstration only.
prescription.add_medicine(medicine)  # Add the medicine.
prescription.send_to_pharmacy()  # Update prescription status.
visit.add_prescription(prescription)  # Link prescription to the visit.

# Create a laboratory order and record its result.
lab_order = LabOrder(601, "Complete Blood Count", 800.00)  # Create lab order.
lab_order.update_status("IN_PROGRESS")  # Mark test in progress.
result = LabResult(701, "Sample result recorded", "N/A", "Refer to lab report")
lab_order.add_result(result)  # Attach the result and complete the order.
visit.add_lab_order(lab_order)  # Link the order to the visit.

# Generate the invoice using consultation, lab, and medicine charges.
invoice = Invoice(
    801, doctor.get_consultation_fee(), 800.00, 0.00, 0.00
)  # Create invoice.
invoice.apply_insurance(500.00)  # Apply example insurance coverage.

# Create a claim for the insurance-covered amount.
claim = InsuranceClaim(901, "Example Health Insurance", 500.00)
claim.submit_claim()  # Submit the simulated claim.
claim.approve_claim(500.00)  # Simulate insurer approval.

# Process the patient's remaining payment.
payment = Payment(1001, invoice.get_total_amount(), "CARD")
payment.process_payment()  # Simulate successful payment.
invoice.set_status("PAID")  # Mark the invoice as paid.

# Print the results.
print("=== ROYAL CARE CLINIC ===")
print("Clinic:", clinic.get_name())
print("Department:", department.get_name())
print("Patient:", patient.get_name())
print("Doctor:", doctor.get_name())
print("Appointment:", appointment.get_status())
print("Appointment time:", appointment.get_appointment_datetime())
print("Visit diagnosis:", visit.get_diagnosis())
print("Vitals:", visit.get_vitals())
print("Prescription:", prescription.get_status())
print("Lab order:", lab_order.get_status())
print("Lab result:", lab_order.get_result().get_result_value())
print(invoice.generate_invoice())
print("Insurance claim:", claim.get_status())
print("Payment:", payment.get_status())
print("Visit summary:", patient.download_visit_summary(401))

=== ROYAL CARE CLINIC ===
Clinic: Royal Care Clinic
Department: General Practice
Patient: Ali
Doctor: Dr. Ahmad
Appointment: COMPLETED
Appointment time: 2026-11-09 10:00:00
Visit diagnosis: Seasonal allergy
Vitals: {'Temperature': '36.8 C', 'Blood Pressure': '118/76 mmHg'}
Prescription: SENT_TO_PHARMACY
Lab order: COMPLETED
Lab result: Sample result recorded
Invoice #801 | Gross: 2300.00 | Insurance: 500.00 | Patient payable: 1800.00 | Status: PAID
Insurance claim: APPROVED
Payment: COMPLETED
Visit summary: Patient reported sneezing and mild nasal symptoms.
